# TCGA-BRCA Multimodal Stratification — 02 · Preprocess

Turn the three raw Xena files into a single **integrated, per-patient matrix** ready for clustering, plus an aligned **metadata** table (survival + PAM50).

Steps, in order:
1. **Load** the raw files
2. **Transpose** the two omics (Xena stores them genes×samples; we need samples×genes)
3. **Keep primary tumours only** (sample-type code `-01`)
4. **Align** — keep patients present in *all three* tables (the honest cost of multimodal integration)
5. **Per modality:** impute → select top-variable genes → z-score → PCA
6. **Integrate** — concatenate the two PCA blocks
7. **Metadata** — attach `OS_time`, `OS_event`, `PAM50`, and save

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

RAW_DIR  = Path("data/raw")
PROC_DIR = Path("data/processed")
PROC_DIR.mkdir(parents=True, exist_ok=True)

# clinical column names we confirmed by listing the matrix in notebook 01
OS_TIME  = "OS_Time_nature2012"
OS_EVENT = "OS_event_nature2012"
PAM50    = "PAM50Call_RNAseq"

# design choices (easy to find and change)
N_TOP_FEATURES = 2000   # most-variable genes kept per modality before standardization and PCA
N_PCS          = 50     # PCA components per modality

## 1 · Load the raw files

The two omics are gzipped TSVs with **genes in rows, samples in columns**; the clinical matrix is plain text with **samples in rows**.

In [2]:
expr_raw = pd.read_csv(RAW_DIR / "expression.gz", sep="\t", index_col=0, compression="gzip")
cnv_raw  = pd.read_csv(RAW_DIR / "cnv.gz",        sep="\t", index_col=0, compression="gzip")
clinical = pd.read_csv(RAW_DIR / "clinical.tsv",  sep="\t", index_col=0)

print("expression :", expr_raw.shape, "(genes x samples)")
print("cnv        :", cnv_raw.shape,  "(genes x samples)")
print("clinical   :", clinical.shape, "(samples x variables)")

expression : (20530, 1218) (genes x samples)
cnv        : (24776, 1080) (genes x samples)
clinical   : (1247, 193) (samples x variables)


## 2 · Transpose the omics

We want one **patient per row**, so the two omics get transposed. The clinical matrix is already in that orientation.

In [3]:
expr = expr_raw.T   # now samples x genes
cnv  = cnv_raw.T     # now samples x genes
print("expression:", expr.shape, "| cnv:", cnv.shape, "(both samples x genes now)")

expression: (1218, 20530) | cnv: (1080, 24776) (both samples x genes now)


## 3 · Keep primary tumours only

A TCGA sample barcode looks like `TCGA-A1-A0SB-01`. The two digits after the third hyphen are the **sample type**: `01` = primary solid tumour, `11` = normal tissue, `06` = metastatic. We keep only `01` so we never mix tumour and normal tissue in the same analysis.

In [4]:
def primary_only(ids):
    """Keep sample IDs whose sample-type code is 01 (primary solid tumour)."""
    keep = []
    for s in ids:
        parts = s.split("-")
        if len(parts) >= 4 and parts[3][:2] == "01":
            keep.append(s)
    return keep

expr = expr.loc[primary_only(expr.index)]
cnv  = cnv.loc[primary_only(cnv.index)]
clin = clinical.loc[primary_only(clinical.index)]
print(f"primary tumours -> expr: {len(expr)}, cnv: {len(cnv)}, clinical: {len(clin)}")

primary tumours -> expr: 1097, cnv: 1080, clinical: 1101


## 4 · Align the samples

This is the heart of honest multimodal integration: we keep only patients measured in **all three** tables. The intersection is smaller than any single table — that's expected, and it's a number to report openly, not hide.

In [5]:
common = sorted(set(expr.index) & set(cnv.index) & set(clin.index))
print(f"common patients across all three modalities: {len(common)}")

expr = expr.loc[common]
cnv  = cnv.loc[common]
clin = clin.loc[common]

common patients across all three modalities: 1078


## 5 · Per-modality processing

For each omic, in this exact order:

1. **Impute** missing values with the per-gene median.
2. **Select** the `N_TOP_FEATURES` most variable genes — *before* scaling. Variance is scale-dependent, so this has to happen on the raw values; if we z-scored first, every gene would have variance 1 and the ranking would be meaningless.
3. **Z-score** the kept genes so all features sit on the same scale.
4. **PCA** to `N_PCS` components — denoises and gives each modality a compact, equal-sized block.

(`n_components` is capped at `samples − 1` and at the number of features, since PCA can't produce more components than either.)

In [17]:
def prep_modality(df: pd.DataFrame, n_top: int, n_pcs: int, name: str):
    """Impute -> select top-variable genes -> z-score -> PCA.
    Returns (scores: samples×PCs, loadings: PCs×genes)."""
    df = df.fillna(df.median())                                   # 1. impute
    top = df.var(axis=0).nlargest(min(n_top, df.shape[1])).index  # 2. select on raw variance
    df = df[top]
    X = StandardScaler().fit_transform(df.values)                 # 3. z-score
    n_comp = min(n_pcs, X.shape[0] - 1, X.shape[1])               # 4. PCA (capped)
    pca = PCA(n_components=n_comp, random_state=42)
    Xp = pca.fit_transform(X)
    pc_names = [f"{name}_PC{i+1}" for i in range(n_comp)]
    scores   = pd.DataFrame(Xp, index=df.index, columns=pc_names)
    loadings = pd.DataFrame(pca.components_, index=pc_names, columns=df.columns)
    print(f"  {name}: {df.shape[1]} genes -> {n_comp} PCs "
          f"({pca.explained_variance_ratio_.sum():.1%} of variance kept)")
    return scores, loadings

print("Reducing each modality:")
expr_pcs, expr_loadings = prep_modality(expr, N_TOP_FEATURES, N_PCS, "expr")
cnv_pcs, cnv_loadings = prep_modality(cnv, N_TOP_FEATURES, N_PCS, "cnv")

Reducing each modality:
  expr: 2000 genes -> 50 PCs (61.1% of variance kept)
  cnv: 2000 genes -> 50 PCs (93.3% of variance kept)


In [19]:
# i 10 geni che pesano di più (in valore assoluto) su PC1 dell'espressione
expr_loadings.loc["expr_PC1"].abs().sort_values(ascending=False).head(10)

sample
FOXA1     0.050829
ESR1      0.050689
TBC1D9    0.049790
FOXC1     0.049543
GATA3     0.049264
MLPH      0.049073
BCL11A    0.048826
VGLL1     0.048635
PSAT1     0.048520
THSD4     0.048472
Name: expr_PC1, dtype: float64

## 6 · Integrate

Concatenate the two PCA blocks side by side. Because we reduced each modality to its **own** PCs first and only then joined them, neither omic dominates just because it had more genes — this is "intermediate" integration, transparent and easy to explain.

In [7]:
integrated = pd.concat([expr_pcs, cnv_pcs], axis=1)
print("integrated matrix:", integrated.shape, "(patients x combined PCs)")

integrated matrix: (1078, 100) (patients x combined PCs)


## 7 · Metadata + save

We pull survival and subtype for the same patients, in the same order as the integrated matrix.

> **Check the survival units.** TCGA times can be in days or months. Look at `OS_time` below: a max around ~4000 means **days**, a max around ~250 means **months**. We'll label the Kaplan–Meier axis accordingly in notebook 03.

In [8]:
meta = clin.loc[common, [OS_TIME, OS_EVENT, PAM50]].rename(
    columns={OS_TIME: "OS_time", OS_EVENT: "OS_event", PAM50: "PAM50"})
meta["OS_time"]  = pd.to_numeric(meta["OS_time"],  errors="coerce")
meta["OS_event"] = pd.to_numeric(meta["OS_event"], errors="coerce")

# alignment guarantees (cheap, catches silent bugs)
assert list(integrated.index) == list(meta.index) == common
assert not integrated.isna().any().any()

print("OS_time range:", meta["OS_time"].min(), "-", meta["OS_time"].max())
print("patients with survival info:", meta[["OS_time", "OS_event"]].dropna().shape[0])
print("PAM50 counts:\n", meta["PAM50"].value_counts(dropna=False))

integrated.to_csv(PROC_DIR / "integrated.csv")
meta.to_csv(PROC_DIR / "metadata.csv")
print("\nsaved -> data/processed/integrated.csv  &  metadata.csv")

OS_time range: 0.0 - 7125.0
patients with survival info: 801
PAM50 counts:
 PAM50
LumA      415
NaN       247
LumB      192
Basal     135
Her2       67
Normal     22
Name: count, dtype: int64

saved -> data/processed/integrated.csv  &  metadata.csv


## Next

`03 · stratify & survival`: UMAP + HDBSCAN on `integrated.csv` (clustering on the integrated PCA space, **not** on the 2D map), then Kaplan–Meier curves per cluster with a log-rank test, and the PAM50 overlap as biological validation.